# Trabajo en clase resuelto - Q-Learning con FrozenLake

**Juan David Sierra Orozco**
Inteligencia Artificial (SI3003), EAFIT

Este es el notebook de clase de la sesión 5 con todas las actividades resueltas: las seis actividades, la comparación "¿qué cambió?" y el cierre de clase. Las respuestas están en celdas de texto marcadas como **Respuesta**, y donde la respuesta se apoya en números hay una celda de código que los calcula.

**Reproducibilidad.** Se fijan las semillas de `random`, `np.random` y del propio ambiente (`env.reset(seed=...)`, `env.action_space.seed(...)`). Las celdas de análisis que agregué corren dentro de `aislado()`, que guarda y restaura el estado de los generadores globales: así una celda de verificación no consume números aleatorios y no cambia lo que ocurre en las celdas siguientes. Las funciones del enunciado (`train_q_learning`, `play_episode`, etc.) están sin modificar.

---

# Trabajo en clase - Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### Semillas y utilidad para aislar el azar

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import contextlib

@contextlib.contextmanager
def aislado():
    """Ejecuta un bloque sin alterar los generadores aleatorios globales.

    Guarda el estado de `random` y de `np.random`, deja correr el bloque y lo
    restaura al salir. Así una celda de verificación o de análisis no cambia
    los números que verán las celdas posteriores."""
    estado_py = random.getstate()
    estado_np = np.random.get_state()
    try:
        yield
    finally:
        random.setstate(estado_py)
        np.random.set_state(estado_np)

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

-
-
-


**Respuesta.** En CheeseWorld escribimos a mano las dos piezas: la clase `Environment` (el mundo que responde con `(r, s')`) y la clase `QLearningAgent` (la Q-table y la regla de actualización). Aquí **Gymnasium ocupa el lugar del ambiente**: `gym.make("FrozenLake-v1")` crea el mundo, `env.reset()` devuelve el estado inicial y `env.step(a)` devuelve `(s', r, terminated, truncated, info)`.

El **agente no es una clase**: quedó repartido en cuatro piezas sueltas que escribimos nosotros. La **Q-table** (`Qtable`, un arreglo NumPy), la **política** (`greedy_policy` y `epsilon_greedy_policy`), la **regla de aprendizaje** (el bloque `td_target`, `td_error` y la suma con `learning_rate` dentro de `train_q_learning`) y el **ciclo de interacción** que conecta todo. Gymnasium sabe cómo es el mundo; la Q-table es lo único que el agente sabe de él.

## 2. Crear FrozenLake


In [3]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset(seed=SEED)
env.action_space.seed(SEED)

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 - La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

-
-
-


**Respuesta.**

1. **16 filas**: una por estado. El mapa 4x4 tiene 16 casillas y cada una es un estado (`env.observation_space.n = 16`).
2. **4 columnas**: una por acción (Left, Down, Right, Up; `env.action_space.n = 4`).
3. **Q[s, a]** es la recompensa total descontada que el agente espera acumular si está en el estado `s`, ejecuta la acción `a` y de ahí en adelante sigue actuando bien. Es el valor de *esa pareja*, no solo del estado: por eso puede decir qué tan buena es cada acción en cada casilla. En total son 16 x 4 = 64 números por aprender.

In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [5]:
print("Filas    (estados) :", Q.shape[0])
print("Columnas (acciones):", Q.shape[1])
print("Celdas que debe aprender el agente:", Q.size)

Filas    (estados) : 16
Columnas (acciones): 4
Celdas que debe aprender el agente: 64


### 💬 Actividad 2 - Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

-
-
-


**Respuesta.** Que todos los valores sean cero **no** significa que todas las acciones sean malas: significa que **el agente todavía no sabe nada**. Cero es un valor de partida, no una evaluación. Una acción solo deja de valer cero cuando el agente la prueba y recibe información (en FrozenLake, la primera recompensa distinta de cero llega recién al alcanzar la meta).

Si varias acciones tienen exactamente el mismo valor máximo, `argmax` por sí solo se quedaría siempre con la primera (Left), y el agente quedaría pegado contra la pared. Por eso `greedy_policy` **desempata al azar** entre las acciones empatadas con `np.flatnonzero(q == max_q)`. La celda de arriba lo comprueba sobre la fila del estado 0: en 2000 decisiones cada acción sale aproximadamente un 25 % de las veces. Con la tabla en cero, la política greedy se comporta como una caminata aleatoria.

In [6]:
# Desempate cuando todas las acciones valen lo mismo (Q = 0).
# Corre aislado para no consumir el generador global.
with aislado():
    elegidas = [greedy_policy(Q, 0) for _ in range(2000)]
    conteo = np.bincount(elegidas, minlength=4)
nombres = ["Left", "Down", "Right", "Up"]
for n, k in zip(nombres, conteo):
    print(f"{n:5s}: {k:4d} de 2000  ({k/2000:.1%})")

Left :  525 de 2000  (26.2%)
Down :  493 de 2000  (24.6%)
Right:  491 de 2000  (24.6%)
Up   :  491 de 2000  (24.6%)


## 3. Ejecutar una transición


In [7]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 0
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 - Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(\quad,\quad,\quad,\quad)
$$

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


In [8]:
nombres = ["Left", "Down", "Right", "Up"]
print(f"Experiencia (s, a, r, s') = ({state}, {action} [{nombres[action]}], {reward}, {next_state})")
print("terminated =", terminated, "| truncated =", truncated)

Experiencia (s, a, r, s') = (0, 0 [Left], 0, 0)
terminated = False | truncated = False


**Respuesta.** Con la semilla fijada, la transición ejecutada fue desde el estado `0` con la acción `0` (Left); el ambiente respondió con recompensa `0` y siguiente estado `0` (el agente chocó contra el borde izquierdo y se quedó en su lugar, como se ve en la celda de arriba). La experiencia es:

$$
(s,a,r,s') = (0,\ 0,\ 0,\ 0)
$$

**Lo que no teníamos directamente en Value Iteration** son los resultados de la acción: la recompensa $r$ y, sobre todo, el siguiente estado $s'$. En Value Iteration conocíamos el modelo completo, $T(s,a,s')$ y $R(s)$, y por eso podíamos *calcular* el valor esperado sobre todos los $s'$ posibles sin ejecutar nada. En Q-Learning el agente no conoce $T$: solo puede actuar y **observar** un $s'$ y un $r$ concretos de cada intento. De la experiencia real, $s$ y $a$ los elige el agente, pero $r$ y $s'$ los entrega el mundo.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [9]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


In [10]:
# Radiografía de las tres tablas (aislado: solo lee, no usa azar).
for nombre, T in [("Q_initial", Q_initial), ("Q_early", Q_early), ("Q_trained", Q_trained)]:
    print(f"{nombre:9s} celdas distintas de cero: {np.count_nonzero(T):2d} de {T.size} | Q máximo: {T.max():.3f}")

Q_initial celdas distintas de cero:  0 de 64 | Q máximo: 0.000
Q_early   celdas distintas de cero:  1 de 64 | Q máximo: 0.700
Q_trained celdas distintas de cero: 35 de 64 | Q máximo: 1.000


### Momento 1 - Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [11]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 - Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [12]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 1.0


### Momento 3 - Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [13]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad - ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


In [14]:
# Política greedy implícita en cada tabla (estado por estado, desempate = primera acción).
FLECHA = {0: "<", 1: "v", 2: ">", 3: "^"}
def mapa_politica(T):
    filas = []
    for r in range(4):
        fila = []
        for c in range(4):
            s = r * 4 + c
            fila.append("." if np.all(T[s] == 0) else FLECHA[int(np.argmax(T[s]))])
        filas.append(" ".join(fila))
    return "\n".join(filas)

for nombre, T in [("Q_early", Q_early), ("Q_trained", Q_trained)]:
    print(nombre, "(. = fila todavía en cero)")
    print(mapa_politica(T)); print()
print("Mapa del lago:  S F F F / F H F H / F F F H / H F F G")

Q_early (. = fila todavía en cero)
. . . .
. . . .
. . . .
. . > .

Q_trained (. = fila todavía en cero)
v > v <
v . v .
> v v .
. > > .

Mapa del lago:  S F F F / F H F H / F F F H / H F F G


**Respuesta.** El ambiente, los estados y las acciones son los mismos en las tres ejecuciones; **lo único que cambió es la Q-table**, el conocimiento interno del agente. Los números de la celda de arriba lo muestran:

- **`Q_initial`** tiene 0 de 64 celdas distintas de cero. El agente no sabe nada, todas las acciones empatan y la política greedy es una caminata al azar.
- **`Q_early`** (50 episodios) tiene 1 sola celda distinta de cero: `Q[14, Right] = 0.7`, la única pareja que alcanzó la meta en esos episodios. Con una sola fila aprendida, en las demás casillas el agente sigue desempatando al azar, así que su desempeño todavía es en gran parte suerte. En esta semilla, justamente, la corrida del Momento 2 llegó a la meta (recompensa 1.0), pero no por tener una política: por el azar de los empates.
- **`Q_trained`** (10 000 episodios) tiene 35 celdas distintas de cero y un máximo de 1.0. La recompensa de la meta se propagó hacia atrás casilla por casilla (por la parte $\gamma\max Q(s',a')$ del TD target), y el mapa de flechas muestra una política coherente que baja y se mueve a la derecha hasta la meta sin pisar los hoyos.

En resumen: aprender es pasar de una tabla vacía a una tabla donde los valores crecen a medida que la acción queda más cerca de la recompensa.

### 💬 Actividad 4 - Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** _______

**Valores Q:**

- Left:
- Down:
- Right:
- Up:

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Interpretación:**

>


In [15]:
# Actividad 4: una fila de Q. Estado 14 = fila 3, columna 2, justo a la izquierda de la meta G (15).
s_elegido = 14
print("Estado seleccionado:", s_elegido)
for n, v in zip(["Left", "Down", "Right", "Up"], Q_trained[s_elegido]):
    print(f"  {n:5s}: {v:.4f}")
print("argmax_a Q(s,a) =", int(np.argmax(Q_trained[s_elegido])))

# Para comparar, una fila lejos de la meta: estado 0 (inicio)
print("\nEstado 0 (inicio):", np.round(Q_trained[0], 4), "-> argmax", int(np.argmax(Q_trained[0])))
print("Estado 6 (casilla F encima de un hoyo):", np.round(Q_trained[6], 4))

Estado seleccionado: 14
  Left : 0.9025
  Down : 0.9500
  Right: 1.0000
  Up   : 0.9025
argmax_a Q(s,a) = 2

Estado 0 (inicio): [0.7351 0.7738 0.7738 0.7351] -> argmax 1
Estado 6 (casilla F encima de un hoyo): [0.     0.9025 0.     0.8145]


**Respuesta.** Estado seleccionado: **14** (fila 3, columna 2 del mapa, justo a la izquierda de la meta `G`).

| Acción | Valor Q |
|---|---:|
| Left | 0.9025 |
| Down | 0.9500 |
| Right | **1.0000** |
| Up | 0.9025 |

$\arg\max_a Q(14,a)=\text{Right}$ (acción 2).

**Interpretación.** Ir a la derecha desde el 14 cae directamente en la meta y entrega recompensa 1, así que $Q(14,\text{Right})=1$. Las demás acciones valen menos porque, aunque todavía se puede llegar a la meta después, hay que dar pasos extra y cada paso descuenta con $\gamma=0.95$: Down vale $0.95=\gamma\cdot 1$ (un paso perdido y luego Right, al estar en la pared inferior el agente se queda en el 14), y Left y Up valen $0.9025=\gamma^2\cdot 1$ (dos pasos). Así se lee la fila: la mejor acción vale exactamente la recompensa, y las otras valen lo mismo descontado por cada paso de más. También se ve en la fila del estado 6 (encima de un hoyo, en la celda de abajo) que las acciones que llevan a un hoyo quedan en 0 y las seguras tienen valor positivo.

## 5. Evaluar la política aprendida


In [16]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


In [17]:
# Para ver qué costaría explorar en evaluación: misma Q, con epsilon = 0.3 (aislado).
with aislado():
    recomp = []
    for _ in range(100):
        s, _ = env.reset()
        total = 0
        for _ in range(100):
            a = epsilon_greedy_policy(Q_trained, s, 0.3, env)
            s, r, term, trunc, _ = env.step(a)
            total += r
            if term or trunc:
                break
        recomp.append(total)
print(f"Evaluación greedy        : {mean_reward:.3f}")
print(f"Evaluación con eps = 0.3 : {np.mean(recomp):.3f}")

Evaluación greedy        : 1.000
Evaluación con eps = 0.3 : 0.680


### 💬 Actividad 5 - Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

>


**Respuesta / conclusión.** Porque en evaluación queremos **medir lo que el agente aprendió**, no seguir aprendiendo. Explorar sirve durante el entrenamiento para descubrir información nueva; una vez terminado, tomar acciones al azar solo mete ruido y castiga el resultado. La celda de arriba lo cuantifica sobre la misma `Q_trained`: con la política greedy la recompensa media es 1.000, y si se explora con $\epsilon=0.3$ baja a 0.680, porque el 30 % de las decisiones son aleatorias y algunas caen en un hoyo. Este es el equilibrio *exploration vs. exploitation*: se explora mientras se aprende y se explota (se usa el argmax) cuando se juega o se evalúa.

## 6. Experimento: FrozenLake estocástico


In [18]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

slippery_env.reset(seed=SEED)
slippery_env.action_space.seed(SEED)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.702
Std reward : 0.457


In [19]:
# Comparación determinístico vs resbaloso (aislado).
with aislado():
    mean_det, std_det = evaluate_q_policy(env, Q_trained, n_episodes=500)
print(f"is_slippery=False: media {mean_det:.3f} | desv {std_det:.3f}")
print(f"is_slippery=True : media {mean_reward:.3f} | desv {std_reward:.3f}")
print()
print("Política aprendida en el ambiente resbaloso:")
print(mapa_politica(Q_slippery))
# Cuántos episodios de entrenamiento terminaron en la meta (últimos 2000)
print(f"\nEntrenamiento resbaloso, éxito en los últimos 2000 episodios: {np.mean(rewards_slippery[-2000:]):.3f}")

is_slippery=False: media 1.000 | desv 0.000
is_slippery=True : media 0.702 | desv 0.457

Política aprendida en el ambiente resbaloso:
< ^ < ^
< . > .
^ v < .
. > v .

Entrenamiento resbaloso, éxito en los últimos 2000 episodios: 0.291


### 💬 Actividad 6 - Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- Ambiente:
- Algoritmo:


**Discusión.**

- **Ambiente.** Con `is_slippery=False` la transición es determinística: la acción elegida siempre se ejecuta. Con `is_slippery=True` el hielo es resbaloso: con probabilidad 1/3 el agente se mueve en la dirección elegida y con 1/3 a cada lado perpendicular, es decir, $T(s,a,s')$ deja de ser 0 o 1 y se reparte entre varios $s'$. El mismo par $(s,a)$ puede dar resultados distintos en intentos distintos.
- **Algoritmo.** **La ecuación de Q-Learning no cambia**: $Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$ es exactamente la misma. Lo que cambia es lo que la actualización *promedia*: como el $s'$ observado varía entre intentos, el TD target es ruidoso y $Q(s,a)$ solo converge a la **esperanza** sobre las transiciones posibles. Eso exige más episodios y una tasa de aprendizaje más cuidadosa.

Los números lo confirman. En el ambiente determinístico la política aprendida llega a la meta el 100 % de las veces (media 1.000, desviación 0.000). En el resbaloso, tras 20 000 episodios, la media de evaluación con la semilla del notebook es 0.702 (desviación 0.457, porque cada episodio termina en 0 o en 1): aun una buena política falla casi un tercio de las veces porque el hielo la desvía. El mapa de flechas aprendido es más conservador que el determinístico: en varias casillas elige moverse *contra* una pared para que el resbalón no la empuje a un hoyo. Y el resultado es **mucho menos estable**: repetido con 5 semillas (última celda de código), la media de evaluación va de 0.158 a 0.748 (promedio 0.519). Con transiciones estocásticas y $lpha=0.7$ cada muestra ruidosa mueve mucho la tabla, así que la calidad de lo aprendido depende de la suerte del entrenamiento; con más episodios o un $lpha$ menor y decreciente se estabilizaría. Mi diagnóstico del $lpha$ es una hipótesis, no algo que haya medido aquí.

In [20]:
# Verificación: ¿qué tan estable es el resultado resbaloso entre semillas? (aislado)
medias = []
with aislado():
    for sem in range(5):
        random.seed(sem); np.random.seed(sem)
        env_s = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="rgb_array")
        env_s.reset(seed=sem); env_s.action_space.seed(sem)
        Qs = initialize_q_table(16, 4)
        Qs, _ = train_q_learning(env_s, Qs, n_episodes=20000, learning_rate=0.7, gamma=0.95,
                                 max_epsilon=1.0, min_epsilon=0.05, decay_rate=0.0005, max_steps=100)
        m, sd = evaluate_q_policy(env_s, Qs, n_episodes=500)
        medias.append(m)
        print(f"semilla {sem}: media de evaluación {m:.3f}")
print(f"\nPromedio entre semillas: {np.mean(medias):.3f} | mín {min(medias):.3f} | máx {max(medias):.3f}")

semilla 0: media de evaluación 0.158


semilla 1: media de evaluación 0.690


semilla 2: media de evaluación 0.352


semilla 3: media de evaluación 0.646


semilla 4: media de evaluación 0.748

Promedio entre semillas: 0.519 | mín 0.158 | máx 0.748


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

>

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

>

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

>

**4. ¿Por qué Q-Learning es model-free?**

>


**Respuestas del cierre de clase.**

**1. ¿Qué almacena $Q(s,a)$?**

> La recompensa total descontada que se espera obtener al estar en el estado $s$, ejecutar la acción $a$ y luego seguir actuando de la mejor manera conocida. Una celda por pareja estado-acción; en FrozenLake son 16 x 4 = 64 valores.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la propia Q-table, de la fila del **siguiente estado** $s'$ que el ambiente devolvió en `env.step`: se toma el mayor de sus valores. Es una estimación del valor futuro que el agente ya tiene guardada (por eso se dice que Q-Learning *bootstrapea*: actualiza una estimación usando otra). Si el episodio terminó, ese término vale 0, como en el código (`best_next_q = 0.0 if done`).

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Porque con la tabla en cero, o con valores aún malos, si el agente siempre explotara se quedaría repitiendo las acciones que ya cree buenas y nunca descubriría rutas mejores o la propia meta. Con probabilidad $\epsilon$ prueba acciones al azar (explora) y con probabilidad $1-\epsilon$ usa el argmax (explota). Se hace decaer $\epsilon$ para explorar mucho al comienzo y explotar cada vez más al final.

**4. ¿Por qué Q-Learning es model-free?**

> Porque nunca usa ni aprende $T(s,a,s')$ ni $R(s)$. Solo usa las experiencias $(s,a,r,s')$ que obtiene al interactuar con el ambiente. Value Iteration, en cambio, necesita el modelo completo para calcular el valor esperado sobre los $s'$ posibles.